<a href="https://colab.research.google.com/github/Alilson2/Projeto_IA/blob/main/GerarDataFrame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr # Ler arquivos netcdf
from google.colab import drive
import seaborn as sns

import matplotlib.patches as mpatches # Desenhar geometria em um mapa

In [ ]:
import os
import xarray as xr

# --- Clonar repositório se não existir ---
if not os.path.exists("Projeto_IA"):
    !git clone https://github.com/Alilson2/Projeto_IA.git
else:
    print("📁 Repositório 'Projeto_IA' já existe — pulando o clone.")

# --- Verificar se a pasta foi criada ---
if not os.path.exists("Projeto_IA"):
    raise FileNotFoundError("❌ A pasta 'Projeto_IA' não foi encontrada. O clone pode ter falhado.")
else:
    print("\n✅ Repositório clonado com sucesso!\n")
    print("Arquivos dentro da pasta Projeto_IA:\n", os.listdir("Projeto_IA"))

# --- Localizar arquivos .nc ---
arquivos_nc = [f for f in os.listdir("Projeto_IA") if f.endswith(".nc")]
if not arquivos_nc:
    raise FileNotFoundError("❌ Nenhum arquivo .nc encontrado na pasta Projeto_IA!")
else:
    print("\n📂 Arquivo(s) NetCDF encontrado(s):")
    for f in arquivos_nc:
        print(" -", f)

# --- Montar lista de caminhos ---
ARQUIVO = [os.path.join("Projeto_IA", f) for f in arquivos_nc]

# --- Função para corrigir longitude ---
def corrigir_longitude(ds):
    for coord in ["longitude", "lon"]:
        if coord in ds.coords:
            ds = ds.assign_coords({coord: ((ds[coord] + 180) % 360) - 180})
            ds = ds.sortby(coord)
    return ds

# --- Abrir arquivos com segurança (nova sintaxe) ---
try:
    dados = xr.open_mfdataset(
        ARQUIVO,
        combine='by_coords',
        parallel=True,           # usa múltiplos núcleos
        preprocess=corrigir_longitude,
        combine_attrs='override' # 🟢 substitui o antigo compat='override'
    )
except ValueError as e:
    print("\n⚠️ Erro na combinação — tentando modo 'nested' (concat por tempo)...")
    dados = xr.open_mfdataset(
        ARQUIVO,
        combine='nested',
        concat_dim='valid_time',  # ajuste se sua dimensão temporal tiver outro nome
        parallel=True,
        preprocess=corrigir_longitude,
        combine_attrs='override'
    )

print("\n✅ Dataset carregado com sucesso!\n")

In [ ]:
df = dados.to_dataframe()
df = df.dropna()

# --- 2️⃣ Verificar se há a coordenada temporal ---
if "valid_time" not in dados.coords:
    raise ValueError("❌ O dataset não contém uma coordenada temporal chamada 'valid_time'.")

# --- 3️⃣ Converter o eixo temporal para pandas.DatetimeIndex ---
tempo = pd.to_datetime(dados["valid_time"].values)

# --- 4️⃣ Criar DataFrame com componentes temporais ---
df_tempo = pd.DataFrame({
    "timestamp": tempo,
    "timestamp_segundos": tempo.view("int64"),   # segundos desde 1970
    "ano": tempo.year,
    "mes": tempo.month,
    "dia": tempo.day,
    "hora": tempo.hour,
    "minuto": tempo.minute,
    "segundo": tempo.second,
    "dia_semana": tempo.dayofweek,
    "dia_do_ano": tempo.dayofyear
})

#print(df_tempo.head())

# Exemplo: seleciona uma variável e um período
# Sort the dataset by valid_time before slicing
dados_sorted = dados.sortby('valid_time')
dados_filtrado = dados_sorted

# Converte para pandas sem estourar RAM
df_panda = dados_filtrado.to_dataframe().reset_index()
df_panda = df_panda.dropna()

# Junta com df_tempo
df_final = pd.merge(
    df_panda,
    df_tempo,
    left_on='valid_time',
    right_on='timestamp',
    how='left'
)


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def criar_dados(df_panda):

    # Criar coluna "data" usando valid_time
    df_panda["data"] = pd.to_datetime(df_panda["valid_time"]).dt.date

    variaveis = ['d2m', 't2m', 'u10', 'v10', 'slhf', 'sshf', 'ssrd',
                 'sp', 'e', 'tp', 'es', 'ev', 'RH', 'VPD']

    grouped = df_panda.groupby(["data", "latitude", "longitude"])

    resultados = {}

    for var in variaveis:

        if var != "tp":
            resultados[f"{var}_mean"] = grouped[var].mean()
            resultados[f"{var}_min"] = grouped[var].min()
            resultados[f"{var}_max"] = grouped[var].max()

        else:
            resultados[f"{var}_sum"] = grouped[var].sum()

    df_diario_pixel = pd.concat(resultados, axis=1).reset_index()
    return df_diario_pixel



def agregar_regional(df_diario_pixel):

    df_diario_pixel['data'] = pd.to_datetime(df_diario_pixel['data'])
    df_diario_pixel['mes']  = df_diario_pixel['data'].dt.month
    grouped = df_diario_pixel.groupby("data")

    resultados = {}

    for col in df_diario_pixel.columns:
        if col in ["data", "latitude", "longitude", "mes"]:
            continue

        if col.endswith("_sum"):
            # Somente TP_SUM → média espacial
            resultados[col] = grouped[col].mean()

        else:
            # Outras variáveis têm várias estatísticas
            resultados[f"{col}_mean"] = grouped[col].mean()
            resultados[f"{col}_min"]  = grouped[col].min()
            resultados[f"{col}_max"]  = grouped[col].max()
            resultados[f"{col}_std"]  = grouped[col].std()

    df_regional = pd.concat(resultados, axis=1).reset_index()

    # Adicionar novamente coluna mês
    df_regional["mes"] = df_regional["data"].dt.month

    return df_regional



def pipeline(df_raw):
    df_coord_diario = criar_dados(df_raw)
    df_final = agregar_regional(df_coord_diario)
    return df_final

def calcula_vapor_umidade(data):
    # Temperatura e ponto de orvalho em Celsius
    t2m_celsius = data["t2m"] - 273.15
    d2m_celsius = data["d2m"] - 273.15

    # 1) Pressão de vapor de saturação (es) em Pa
    es = 610.94 * np.exp(17.625 * t2m_celsius / (243.04 + t2m_celsius))

    # 2) Pressão de vapor real (ev) em Pa
    ev = 610.94 * np.exp(17.625 * d2m_celsius / (243.04 + d2m_celsius))

    # 3) Umidade Relativa (%)
    RH = 100 * ev / es

    # 4) Déficit de Pressão de Vapor (kPa)
    VPD = (es - ev) / 1000

    # Retorna como DataFrame para fácil concatenação
    return pd.DataFrame({
        "es": es,
        "ev": ev,
        "RH": RH,
        "VPD": VPD
    })


In [ ]:
df_panda = df_final
# --- Aplicar ao DataFrame original ---
resultados = calcula_vapor_umidade(df_panda)

# Adiciona novas colunas ao df_panda
df_panda = pd.concat([df_panda.reset_index(drop=True), resultados.reset_index(drop=True)], axis=1)

O_lim = -46.84
L_lim = -46.36
N_lim = -23.4
S_lim = -23.74

ind = np.where((df_panda['latitude'] >= S_lim) & (df_panda['latitude'] <= N_lim) & (df_panda['longitude'] <= L_lim) & (df_panda['longitude'] >= O_lim))[0]
df_panda = df_panda.iloc[ind].reset_index()

In [ ]:
df_diario = pipeline(df_panda)